In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append('/tmp/local_scratch/v_neelesh_bisht/3d-cnn/rsna')

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
from math import ceil
import matplotlib.patches as patches

from resnet_cams import ResnetCAMS
from dataset.dataset import TrainAndValidateDataset

In [ ]:
device = torch.device('cuda:7' if torch.cuda.is_available() else 'cpu')

In [ ]:
train_and_validate_dataset = TrainAndValidateDataset(test_size=0.03, val_size=0.03)

train_loader = train_and_validate_dataset.train_loader
val_loader = train_and_validate_dataset.val_loader
test_loader = train_and_validate_dataset.test_loader


# train_labels = train_and_validate_dataset.train_labels
val_labels = train_and_validate_dataset.val_labels
test_labels = train_and_validate_dataset.test_labels

# print(train_labels.shape)
print("validation size :",val_labels.shape)
print("test size :",test_labels.shape)

# print(f'patientId: {train_labels[0][0]}, Target: {train_labels[0][1]}')

# train_paths = train_and_validate_dataset.train_paths
# val_paths = train_and_validate_dataset.val_paths

# print(len(train_paths))
# print(len(val_paths))

# train_and_validate_dataset.show_raw_dataset()

# train_and_validate_dataset.show_train_dataset()

# train_and_validate_dataset.show_dataloader()

# train_dataset = train_and_validate_dataset.train_dataset
# image = iter(train_dataset)
# img, label, box = next(image)
# print(img.shape)
# print(label, box)
# img = np.transpose(img, (1, 2, 0))
# plt.imshow(img)    

In [ ]:

last_conv_layer_name = 'layer4'

target_class_idx = 1  # Target class
ref_class_idx = 0  # Reference class

resnet_model = torch.load('./rsna-dataset/model.pth')
resnet_model.to(device)


In [ ]:
final_label_arr = [] # value for all_labels
final_feature_arr = [] # value for all_features

embedding = None

def _embedding_hook_fn(module, input, output):
    global embedding 
    embedding = input[0]  # Storing the input to the fc layer
    # print("inside _embedding_hook_fn: ",input[0].shape) ## torch.Size([128, 512])

embedding_hook = resnet_model.fc.register_forward_hook(_embedding_hook_fn)

resnet_model.eval()

correct = 0
total = 0  
for images, labels, _ in tqdm(val_loader):
    images = images.to(device)
    labels = labels.to(device)
    predictions = resnet_model(images)
    _, predicted = torch.max(predictions, 1)
    total += labels.size(0)
    correct += (labels == predicted).sum()

    for i in range(images.shape[0]):
        class_id = labels[i].cpu().item()
        final_label_arr.append(class_id)

        feature = embedding[i].detach().cpu().numpy()
        if len(final_feature_arr) == 0:
            final_feature_arr = np.expand_dims(feature, axis=0)
        else:
            final_feature_arr = np.concatenate((final_feature_arr, np.expand_dims(feature, axis=0)), axis = 0)

print(f'Val_Acc: {100*correct/total}')

embedding_hook.remove()

In [ ]:
####################################### Calculate IOU #######################################
Orig_img_size = 1024
img_size = 224

heatmap_iou_obj = {
        "grad_cam_heatmap":[], 
        "diff_grad_cam_heatmap": [],
        "diff_cam_heatmap": [], 
        "counter_factual_heatmap": [],
        "torch_cam_grad_cam_heatmap": [],
        "torch_cam_grad_campp_heatmap": [],
        "torch_cam_score_cam_heatmap": []
    }

def plot_heatmap_and_mask(binary_heatmap, bbox_mask):
    """Plot binary heatmap and bounding box mask side by side."""
    
    # Plot heatmap and mask
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    
    # Plot binary heatmap
    ax[0].imshow(binary_heatmap, cmap='jet')
    ax[0].set_title('Binary Heatmap')
    ax[0].axis('off')
    
    # Plot bbox mask
    ax[1].imshow(bbox_mask, cmap='jet')
    ax[1].set_title('Bounding Box Mask')
    ax[1].axis('off')
    
    plt.tight_layout()
    plt.show()

def bbox_to_mask(bbox, shape):
    """Convert bounding box coordinates to binary mask."""
    x, y, width, height = bbox

    x = ceil(x*img_size/Orig_img_size) if not np.isnan(bbox[0]) else 0
    y = ceil(y*img_size/Orig_img_size) if not np.isnan(bbox[1]) else 0
    width = ceil(width*img_size/Orig_img_size) if not np.isnan(bbox[2]) else 0
    height = ceil(height*img_size/Orig_img_size) if not np.isnan(bbox[3]) else 0

    x_min = x
    y_min = y
    x_max = x + width
    y_max = y + height
    
    # Create an empty mask
    mask = np.zeros(shape, dtype=np.uint8)
    
    # Ensure the coordinates are within bounds
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(shape[1], x_max)
    y_max = min(shape[0], y_max)
    
    # Draw the bounding box on the mask
    mask[y_min:y_max, x_min:x_max] = 1
    
    return mask

def compute_iou(bbox, heatmap, threshold=63):
    """Calculate the 2D IoU between bounding box and heatmap."""
    # Convert heatmap to binary mask using the threshold
    # print("heatmap", np.max(heatmap), np.min(heatmap))
    # print("bbox", bbox)
    binary_heatmap = (heatmap > threshold).astype(np.uint8)
    
    # Get the bounding box as a binary mask
    bbox_mask = bbox_to_mask(bbox, binary_heatmap.shape)
    
    # print("shape",binary_heatmap.shape, bbox_mask.shape )
    # plot_heatmap_and_mask(binary_heatmap, bbox_mask)
    # Compute intersection and union
    intersection = np.logical_and(binary_heatmap, bbox_mask).sum()
    union = np.logical_or(binary_heatmap, bbox_mask).sum()
    
    # Calculate IoU
    iou = intersection / union if union != 0 else 0
    # print("iou",iou)
    return iou

In [ ]:
final_image_arr = []
final_bbox_arr = []
final_heatmap_arr = {
    "grad_cam_heatmap": [],
    "diff_grad_cam_heatmap": [],
    "diff_cam_heatmap": [],
    "counter_factual_heatmap": [],
    "torch_cam_grad_cam_heatmap": [],
    "torch_cam_grad_campp_heatmap": [],
    "torch_cam_score_cam_heatmap": []
}

for images, labels, bboxs in tqdm(test_loader):
    images = images.to(device)
    labels = labels.to(device)
    
    # Iterate over each image in the batch
    for i in range(images.size(0)):
        # Select a single image and label
        image = images[i:i+1]  # Keep batch dimension (1, C, H, W)
        label = labels[i:i+1].cpu().item()

        bbox_coords = []
        for idx, bbox in enumerate(bboxs):
            # Convert tensor to a list or numpy array if needed
            coords_arr = bbox.tolist() 
            bbox_coords.append(coords_arr[i])

        # filter the images with atleast one bounding box for calculating IOU.
        if np.isnan(bbox_coords[0]) or np.isnan(bbox_coords[1]) or np.isnan(bbox_coords[2]) or np.isnan(bbox_coords[3]):
            continue
        
        bbox_coords = [int(x) for x in bbox_coords]

        final_image_arr.append(image)
        final_bbox_arr.append(bbox_coords)

        cams = ResnetCAMS.get_cams(image, resnet_model, last_conv_layer_name, target_class_idx, ref_class_idx, final_feature_arr=final_feature_arr, final_label_arr=final_label_arr)

        final_heatmap_arr["grad_cam_heatmap"].append(cams["grad_cam_heatmap"])
        final_heatmap_arr["diff_grad_cam_heatmap"].append(cams["diff_grad_cam_heatmap"])
        final_heatmap_arr["diff_cam_heatmap"].append(cams["diff_cam_heatmap"])
        final_heatmap_arr["counter_factual_heatmap"].append(cams["counter_factual_heatmap"])
        final_heatmap_arr["torch_cam_grad_cam_heatmap"].append(cams["torch_cam_grad_cam_heatmap"])
        final_heatmap_arr["torch_cam_grad_campp_heatmap"].append(cams["torch_cam_grad_campp_heatmap"])
        final_heatmap_arr["torch_cam_score_cam_heatmap"].append(cams["torch_cam_score_cam_heatmap"])

        heatmap_iou_obj["grad_cam_heatmap"].append(compute_iou(bbox_coords, cams["grad_cam_heatmap"]))
        heatmap_iou_obj["diff_grad_cam_heatmap"].append(compute_iou(bbox_coords, cams["diff_grad_cam_heatmap"]))
        heatmap_iou_obj["diff_cam_heatmap"].append(compute_iou(bbox_coords, cams["diff_cam_heatmap"]))
        heatmap_iou_obj["counter_factual_heatmap"].append(compute_iou(bbox_coords, cams["counter_factual_heatmap"]))
        heatmap_iou_obj["torch_cam_grad_cam_heatmap"].append(compute_iou(bbox_coords, cams["torch_cam_grad_cam_heatmap"]))
        heatmap_iou_obj["torch_cam_grad_campp_heatmap"].append(compute_iou(bbox_coords, cams["torch_cam_grad_campp_heatmap"]))
        heatmap_iou_obj["torch_cam_score_cam_heatmap"].append(compute_iou(bbox_coords, cams["torch_cam_score_cam_heatmap"]))



In [ ]:
print(len(final_image_arr))
print(final_image_arr[0].shape)
print(final_bbox_arr[0])

In [ ]:
numrows = len(final_image_arr)

# Create a single figure with subplots
fig, ax = plt.subplots(numrows, 8, figsize=(30, 5 * (numrows + 1)))

# Loop through each slice and plot the images in the appropriate subplot
for slice_idx in range(0, numrows):

    # Adding titles for the first row
    if slice_idx == 0:
        ax[slice_idx, 0].set_title('Sample')
        ax[slice_idx, 1].set_title('Mask')
        ax[slice_idx, 2].set_title('Grad-CAM')
        ax[slice_idx, 3].set_title('Diff Grad-CAM')
        ax[slice_idx, 4].set_title('Diff-CAM')
        ax[slice_idx, 5].set_title('Counter Factual')
        ax[slice_idx, 6].set_title('Torch-CAM Grad-CAM')
        ax[slice_idx, 7].set_title('Torch-CAM Grad-CAM++')

    final_image = final_image_arr[slice_idx]
    final_image = final_image.cpu().numpy()
    final_image = final_image[0]
    img = np.transpose(final_image, (1, 2, 0))

    box = final_bbox_arr[slice_idx]
    # 'r' means relative. 'c' means center.
    rx = ceil(box[0]*img_size/Orig_img_size) if not np.isnan(box[0]) else 0
    ry = ceil(box[1]*img_size/Orig_img_size) if not np.isnan(box[1]) else 0
    rw = ceil(box[2]*img_size/Orig_img_size) if not np.isnan(box[2]) else 0
    rh = ceil(box[3]*img_size/Orig_img_size) if not np.isnan(box[3]) else 0

    grad_cam_heatmap = final_heatmap_arr["grad_cam_heatmap"][slice_idx]
    diff_grad_cam_heatmap = final_heatmap_arr["diff_grad_cam_heatmap"][slice_idx]
    diff_cam_heatmap = final_heatmap_arr["diff_cam_heatmap"][slice_idx]
    counter_factual_heatmap = final_heatmap_arr["counter_factual_heatmap"][slice_idx]
    torch_cam_grad_cam_heatmap = final_heatmap_arr["torch_cam_grad_cam_heatmap"][slice_idx]
    torch_cam_grad_campp_heatmap = final_heatmap_arr["torch_cam_grad_campp_heatmap"][slice_idx]
    # grad_cam_heatmap = final_heatmap_arr["grad_cam_heatmap"][slice_idx]

    ax[slice_idx, 0].imshow(img, origin='upper', cmap='bone')
    ax[slice_idx, 0].set_xlabel(f'Image {slice_idx + 1}')

    # TODO: FIT SCORE CAM HERE
    # ax[slice_idx, 1].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))

    img3 = ax[slice_idx, 2].imshow(img, cmap='bone')
    img4 = ax[slice_idx, 2].imshow(grad_cam_heatmap, cmap='jet', alpha=0.5, extent=img3.get_extent())
    img5 = ax[slice_idx, 2].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 2].set_xlabel(f'Grad-CAM {slice_idx + 1}, iou: {heatmap_iou_obj["grad_cam_heatmap"][slice_idx]}')

    img6 = ax[slice_idx, 3].imshow(img, cmap='bone')
    img7 = ax[slice_idx, 3].imshow(diff_grad_cam_heatmap, cmap='jet', alpha=0.5, extent=img6.get_extent())
    img8 = ax[slice_idx, 3].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 3].set_xlabel(f'Diff Grad-CAM {slice_idx + 1}, iou: {heatmap_iou_obj["diff_grad_cam_heatmap"][slice_idx]}')

    img9 = ax[slice_idx, 4].imshow(img, cmap='bone')
    img10 = ax[slice_idx, 4].imshow(diff_cam_heatmap, cmap='jet', alpha=0.5, extent=img9.get_extent())
    img11 = ax[slice_idx, 4].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 4].set_xlabel(f'Diff-CAM {slice_idx + 1}, iou: {heatmap_iou_obj["diff_cam_heatmap"][slice_idx]}')

    img12 = ax[slice_idx, 5].imshow(img, cmap='bone')
    img13 = ax[slice_idx, 5].imshow(counter_factual_heatmap, cmap='jet', alpha=0.5, extent=img12.get_extent())
    img14 = ax[slice_idx, 5].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 5].set_xlabel(f'Counter Factual {slice_idx + 1}, iou: {heatmap_iou_obj["counter_factual_heatmap"][slice_idx]}')

    img15 = ax[slice_idx, 6].imshow(img, cmap='bone')
    img16 = ax[slice_idx, 6].imshow(torch_cam_grad_cam_heatmap, cmap='jet', alpha=0.5, extent=img15.get_extent())
    img17 = ax[slice_idx, 6].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 6].set_xlabel(f'Torch-CAM Grad-CAM {slice_idx + 1}, iou: {heatmap_iou_obj["torch_cam_grad_cam_heatmap"][slice_idx]}')

    img18 = ax[slice_idx, 7].imshow(img, cmap='bone')
    img19 = ax[slice_idx, 7].imshow(torch_cam_grad_campp_heatmap, cmap='jet', alpha=0.5, extent=img18.get_extent())
    img20 = ax[slice_idx, 7].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 7].set_xlabel(f'Torch-CAM Grad-CAM++ {slice_idx + 1}, iou: {heatmap_iou_obj["torch_cam_grad_campp_heatmap"][slice_idx]}')

# Adjust the layout
plt.tight_layout()
plt.show()

In [ ]:
def compute_mean_iou_for_heatmaps(heatmap_iou_obj):
    """Calculate the mean IoU for each type of heatmap."""
    mean_iou = {}

    # Iterate over each type of heatmap
    for heatmap_type, iou_list in heatmap_iou_obj.items():
        # Check if the IoU list is not empty
        if iou_list:
            mean_iou[heatmap_type] = np.mean(iou_list) 
        else:
            mean_iou[heatmap_type] = None  # Use None to indicate no IoU values available
    
    return mean_iou

mean_iou = compute_mean_iou_for_heatmaps(heatmap_iou_obj)


for heatmap_type, iou in mean_iou.items():
    print(f"IOU for {heatmap_type}: ", iou*100,'%')


In [ ]:
# Here’s a more detailed color mapping from the jet colormap:

# Blue: Low values (0 to ~0.25) (0 to 63)
# Cyan: Mid-low values (~0.25 to ~0.45) (63 to 114)
# Green: Mid values (~0.45 to ~0.55) > (~114 to ~140)
# Yellow: High intermediate values (~0.55 to ~0.75)
# Orange: Very high intermediate values (~0.75 to ~0.85)
# Red: Maximum values (~0.85 to 1)

#ASSUMPTIONS of threshold here
HEATMAP_THRESHOLD = 114

